# Daily Challenge — MCP + Airbnb (Student, completed)

A tiny MCP mini-agent that combines:

- **A local notes MCP server** you run yourself (`add_note`, `list_notes`)
- **An Airbnb MCP server** — the real `@openbnb/mcp-server-airbnb`, or a built-in **stub** that returns fixed fake listings (no npm/keys needed)
- **A planner** — a rule-based **stub planner** (default, no API key needed) or a **real LLM** via GitHub Models (Azure Inference SDK, `gpt-4o`)

Flow: connect both MCP servers → list their tool schemas → planner decides `tool_calls` → execute each call against the right server → (optionally) have an LLM summarize the results.

Defaults are set so the whole notebook runs **end-to-end with zero external services**. Flip `USE_REAL_AIRBNB` / `USE_REAL_LLM` on once you have npm / a `GITHUB_TOKEN`.


In [ ]:
# Install (run once). npm line only matters if you set USE_REAL_AIRBNB = True below.
!pip install -q mcp nest_asyncio requests
!pip install -q azure-ai-inference

# Optional: real Airbnb MCP server (only needed for USE_REAL_AIRBNB = True)
!npm install -g @openbnb/mcp-server-airbnb

In [ ]:
import sys
from ipykernel.iostream import OutStream

def _patched_fileno(self):
    # stdout -> 1, stderr -> 2
    if self is sys.stderr:
        return 2
    return 1

# Patch the class for all OutStream instances
OutStream.fileno = _patched_fileno

# And patch the current instances explicitly
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2

## Config

Toggle `USE_REAL_AIRBNB` / `USE_REAL_LLM` when you have npm / a GitHub token. **Defaults below run fully stubbed** — nothing external required.


In [ ]:
import os
from pathlib import Path

MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_AIRBNB = False  # set True if you ran: npm install -g @openbnb/mcp-server-airbnb
USE_REAL_LLM = False     # set True if you have a GITHUB_TOKEN (GitHub Models / Azure Inference)

BASE_ENV = os.environ.copy()
BASE_ENV["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN

In [ ]:
# Only needed if USE_REAL_LLM = True. Safe to run either way.
if USE_REAL_LLM:
    from google.colab import userdata  # Colab secrets API

    # If your secret is saved under the key "GITHUB_TOKEN" in Colab:
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

    # If you used a different key name in the secrets UI, e.g. "github_token":
    # os.environ["GITHUB_TOKEN"] = userdata.get("github_token")

    print("GITHUB_TOKEN visible to Python:", bool(os.getenv("GITHUB_TOKEN")))
else:
    print("USE_REAL_LLM is False -> running with the stub planner, no GITHUB_TOKEN needed.")

## Local notes MCP server

In [ ]:
LOCAL_SERVER = Path("local_notes_server.py")
LOCAL_SERVER.write_text(
'''from mcp.server.fastmcp import FastMCP

notes = []
mcp = FastMCP("local-notes")

@mcp.tool()
def add_note(text: str) -> str:
    """Add a note to the in-memory list."""
    notes.append(text)
    return f"Saved note #{len(notes)}: {text}"

@mcp.tool()
def list_notes() -> str:
    """List saved notes."""
    if not notes:
        return "No notes yet"
    return "\\n".join(f"{i+1}. {n}" for i, n in enumerate(notes))

if __name__ == "__main__":
    mcp.run()
'''.strip() + "",
    encoding="utf-8",
)
print("wrote", LOCAL_SERVER)

## Stub Airbnb MCP server

Used automatically whenever `USE_REAL_AIRBNB = False`. Returns fixed fake listings so the pipeline runs with zero external dependencies. When `USE_REAL_AIRBNB = True`, this file is written but ignored — `orchestrate()` launches the real `npx @openbnb/mcp-server-airbnb` instead.


In [ ]:
STUB_AIRBNB_SERVER = Path("stub_airbnb_server.py")
STUB_AIRBNB_SERVER.write_text(
'''from mcp.server.fastmcp import FastMCP

mcp = FastMCP("stub-airbnb")

_FAKE_LISTINGS = {
    "default": [
        {"name": "Cozy Studio Near Center", "price_per_night": 62, "rating": 4.8},
        {"name": "Sunny 1BR With Balcony", "price_per_night": 89, "rating": 4.6},
        {"name": "Budget Room In Shared Flat", "price_per_night": 34, "rating": 4.2},
    ]
}

@mcp.tool()
def search_listings(city: str) -> str:
    """Search fake Airbnb-style listings for a city (stub data)."""
    listings = _FAKE_LISTINGS.get(city.lower(), _FAKE_LISTINGS["default"])
    lines = [f"Fake listings for {city}:"]
    for l in listings:
        lines.append(f"- {l[\'name\']} - ${l[\'price_per_night\']}/night, {l[\'rating\']} stars")
    return "\\n".join(lines)

if __name__ == "__main__":
    mcp.run()
'''.strip() + "",
    encoding="utf-8",
)
print("wrote", STUB_AIRBNB_SERVER)

## Client helpers (convert tools, stub/real planner, stub/real answer LLM)

In [ ]:
import asyncio
import json
import re
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

def convert_tool(tool, prefix: str):
    # Azure requires ^[a-zA-Z0-9_\.-]+$, so no slashes
    fn_name = f"{prefix}__{tool.name}"
    return {
        "type": "function",
        "function": {
            "name": fn_name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }

In [ ]:
def _extract_city(prompt: str) -> str:
    """Very small heuristic: grabs the capitalized word after 'in'."""
    m = re.search(r"\bin ([A-Z][a-zA-Z]+)", prompt)
    return m.group(1) if m else "Paris"

def _extract_note_text(prompt: str) -> str:
    """Very small heuristic: grabs text after 'note' / 'note that' / 'note:'."""
    m = re.search(r"note(?: that)?[:\-]?\s*(.+)", prompt, re.IGNORECASE)
    return m.group(1).strip() if m else prompt


def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    """Decide which tool_calls to run.

    use_real=False -> rule-based stub planner (no API key needed).
    use_real=True  -> real LLM (gpt-4o via GitHub Models / Azure Inference SDK)
                      picks tool_calls using function calling.
    """
    if not use_real:
        lowered = prompt.lower()
        calls = []
        for f in functions:
            name = f["function"]["name"]
            if name.endswith("__list_notes") and "note" in lowered:
                calls.append({"name": name, "args": {}})
            elif name.endswith("__add_note") and "note" in lowered and (
                "add" in lowered or "save" in lowered
            ):
                calls.append({"name": name, "args": {"text": _extract_note_text(prompt)}})
            elif name.endswith("__search_listings") and any(
                k in lowered for k in ["airbnb", "listing", "stay", " in "]
            ):
                calls.append({"name": name, "args": {"city": _extract_city(prompt)}})

        if not calls:
            # Fallback: show what we can do — list notes + search a default city.
            for f in functions:
                name = f["function"]["name"]
                if name.endswith("__list_notes"):
                    calls.append({"name": name, "args": {}})
                elif name.endswith("__search_listings"):
                    calls.append({"name": name, "args": {"city": "Paris"}})
        return calls

    # ---- real LLM path ----
    import os
    from azure.ai.inference import ChatCompletionsClient
    from azure.ai.inference.models import ChatCompletionsToolDefinition, FunctionDefinition
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use the stub planner (use_real=False).")

    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com", AzureKeyCredential(token)
    )

    tools = [
        ChatCompletionsToolDefinition(
            function=FunctionDefinition(
                name=f["function"]["name"],
                description=f["function"]["description"],
                parameters=f["function"]["parameters"],
            )
        )
        for f in functions
    ]

    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        tools=tools,
        tool_choice="auto",
        temperature=0,
    )

    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls

In [ ]:
def answer_with_llm(
    user_prompt: str,
    tool_calls: List[Dict[str, Any]],
    tool_results: List[Dict[str, Any]],
    use_real: bool = True,
) -> str:
    # Shrink tool_results before sending anywhere / printing.
    small_results = []
    for r in tool_results:
        content = r.get("content", [])
        short_content = []
        if content:
            first = content[0]
            if isinstance(first, str) and len(first) > 4000:
                first = first[:4000] + "...(truncated)..."
            short_content = [first]
        small_results.append(
            {"name": r.get("name"), "args": r.get("args", {}), "content": short_content}
        )

    if not use_real:
        # Stub summarizer: no LLM call, just formats the tool outputs as markdown.
        lines = [f"### Answer for: {user_prompt}", ""]
        for r in small_results:
            content_str = "\n".join(r["content"]) if r["content"] else "(no output)"
            lines.append(f"**{r['name']}**  args={r['args']}")
            lines.append(content_str)
            lines.append("")
        lines.append("## Tools used")
        for name in dict.fromkeys(r["name"] for r in small_results):
            lines.append(f"- {name}")
        return "\n".join(lines)

    # ---- real LLM path ----
    import os
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use_real=False in answer_with_llm.")

    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com",
        AzureKeyCredential(token),
    )

    payload = {
        "user_question": user_prompt,
        "tool_calls": tool_calls,
        "tool_results": small_results,
    }

    resp = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer the user's question using the given tool outputs.\n"
                    "JSON contains user_question, tool_calls, and tool_results (already truncated).\n"
                    "1. Answer clearly in markdown.\n"
                    "2. At the end, add:\n"
                    "## Tools used\n"
                    "- One bullet per distinct tool name.\n"
                ),
            },
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            },
        ],
        temperature=0,
        max_tokens=600,
    )

    msg = resp.choices[0].message
    parts = getattr(msg, "content", None)
    if isinstance(parts, list):
        texts = []
        for p in parts:
            text = getattr(p, "text", None) or getattr(p, "content", None)
            if isinstance(text, str):
                texts.append(text)
        if texts:
            return "".join(texts)

    return str(msg.content)

## Orchestrate (connect both servers and execute tool_calls)

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def orchestrate(prompt: str):
    local_params = StdioServerParameters(
        command="mcp",
        args=["run", str(LOCAL_SERVER)],
        env=BASE_ENV,
    )

    if USE_REAL_AIRBNB:
        airbnb_params = StdioServerParameters(
            command="npx",
            args=["@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
            env=BASE_ENV,
        )
    else:
        # Stub server: plain python process, no npm/network needed.
        airbnb_params = StdioServerParameters(
            command=sys.executable,
            args=[str(STUB_AIRBNB_SERVER)],
            env=BASE_ENV,
        )

    async with stdio_client(local_params) as (lr, lw):
        async with ClientSession(lr, lw) as local_sess:
            await local_sess.initialize()
            local_tools = await local_sess.list_tools()

            async with stdio_client(airbnb_params) as (ar, aw):
                async with ClientSession(ar, aw) as airbnb_sess:
                    await airbnb_sess.initialize()
                    airbnb_tools = await airbnb_sess.list_tools()

                    functions = (
                        [convert_tool(t, "notes") for t in local_tools.tools]
                        + [convert_tool(t, "airbnb") for t in airbnb_tools.tools]
                    )

                    tool_calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
                    print("tool_calls:", tool_calls)

                    tool_results = []
                    for call in tool_calls:
                        name = call["name"]
                        args = call["args"]
                        prefix, tool_name = name.split("__", 1)

                        if prefix == "notes":
                            res = await local_sess.call_tool(tool_name, args)
                            tool_results.append(
                                {
                                    "name": name,
                                    "args": args,
                                    "content": [c.text for c in res.content if hasattr(c, "text")],
                                }
                            )
                        elif prefix == "airbnb":
                            res = await airbnb_sess.call_tool(tool_name, args)
                            payload = []
                            if hasattr(res, "content"):
                                for c in res.content:
                                    if hasattr(c, "text"):
                                        payload.append(c.text)
                            tool_results.append(
                                {
                                    "name": name,
                                    "args": args,
                                    "content": payload,
                                }
                            )

                    return tool_calls, tool_results

## Demo

Adjust the prompt as you like. Switch `USE_REAL_AIRBNB` / `USE_REAL_LLM` to `True` above when you're ready to run against the real Airbnb MCP server and/or a real LLM.


In [ ]:
prompt = "List my notes, find Airbnb listings in Paris, and add a note that I searched for a trip to Paris."

tool_calls, tool_results = asyncio.run(orchestrate(prompt))

print("=== Tool calls ===")
print(json.dumps(tool_calls, indent=2))

print("\n=== Tool results ===")
print(json.dumps(tool_results, indent=2))

final_answer = answer_with_llm(prompt, tool_calls, tool_results, use_real=USE_REAL_LLM)
print("\n=== Final answer ===")
print(final_answer)